In [ ]:
%%capture
!pip install timm==0.9.16 loguru omegaconf torchreid opencv-python-headless

import os
import sys
import time
import json
from collections import defaultdict

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from PIL import Image
import torchvision.transforms as T
import torchvision.models as tv_models
from loguru import logger

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"PyTorch: {torch.__version__}")

In [ ]:
CFG = {
    "dataset_root": "/kaggle/working/cityflowv2_reid",
    "weights_output": "/kaggle/working/resnet101ibn_cityflowv2_384px_best.pth",
    "checkpoint_dir": "/kaggle/working/checkpoints",
    "backbone": "resnet101_ibn_a",
    "feat_dim": 2048,
    "img_size": (384, 384),
    "gem_p": 3.0,
    "epochs": 80,
    "batch_size": 48,
    "num_instances": 4,
    "lr": 3.5e-4,
    "warmup_epochs": 10,
    "eta_min": 1e-6,
    "weight_decay": 5e-4,
    "label_smoothing": 0.1,
    "triplet_margin": 0.3,
    "circle_m": 0.25,
    "circle_gamma": 80,
    "triplet_weight": 1.0,
    "circle_weight": 1.0,
    "id_weight": 1.0,
    "random_erasing_prob": 0.5,
    "color_jitter": True,
    "eval_every": 10,
    "fp16": True,
}

os.makedirs(CFG["checkpoint_dir"], exist_ok=True)
print(json.dumps(CFG, indent=2, default=str))

In [ ]:
# Build ReID crops from raw CityFlowV2 videos
import shutil
from pathlib import Path
from collections import defaultdict

POSSIBLE_ROOTS = [
    Path("/kaggle/input/data-aicity-2023-track-2"),
    Path("/kaggle/input/datasets/thanhnguyenle/data-aicity-2023-track-2"),
]
MAX_SAMPLES_PER_TRACK = 15
MIN_BOX_AREA = 2000
MIN_BOX_SIDE = 30

DATA_ROOT = None
for root in POSSIBLE_ROOTS:
    if root.exists():
        DATA_ROOT = root
        break

if DATA_ROOT is None:
    raise FileNotFoundError(f"CityFlowV2 dataset not found at any of: {POSSIBLE_ROOTS}")

print(f"Dataset root: {DATA_ROOT}")
print(f"Contents: {[p.name for p in DATA_ROOT.iterdir()]}")

CROP_ROOT = Path("/kaggle/working/cityflowv2_crops")
if CROP_ROOT.exists():
    shutil.rmtree(CROP_ROOT)
CROP_ROOT.mkdir(parents=True)

camera_dirs = []
for split_name in ["train", "validation", "test"]:
    split_dir = DATA_ROOT / split_name
    if split_dir.exists():
        for scene_dir in sorted(split_dir.glob("S*")):
            for cam_dir in sorted(scene_dir.glob("c*")):
                camera_dirs.append((split_name, scene_dir.name, cam_dir.name, cam_dir))

print(f"Found {len(camera_dirs)} cameras: {[(s, sc, c) for s, sc, c, _ in camera_dirs]}")

all_crops = {}

for split_name, scene, cam, cam_dir in camera_dirs:
    cam_id = f"{scene}_{cam}"

    gt_file = None
    for gt_path in [cam_dir / "gt" / "gt.txt", cam_dir / "gt.txt"]:
        if gt_path.exists():
            gt_file = gt_path
            break

    if gt_file is None:
        print(f"  Skip {cam_id}: no GT file found")
        continue

    video_file = None
    for ext in ["*.mp4", "*.avi", "*.mkv"]:
        videos = sorted(cam_dir.glob(ext))
        if videos:
            video_file = videos[0]
            break
    if video_file is None:
        for vdo_path in sorted(cam_dir.glob("vdo.*")):
            video_file = vdo_path
            break

    if video_file is None:
        print(f"  Skip {cam_id}: no video file found")
        continue

    detections = defaultdict(list)
    with gt_file.open("r", encoding="utf-8") as handle:
        for line in handle:
            parts = line.strip().split(",")
            if len(parts) < 6:
                continue
            frame_id = int(parts[0])
            track_id = int(parts[1])
            x = int(float(parts[2]))
            y = int(float(parts[3]))
            w = int(float(parts[4]))
            h = int(float(parts[5]))
            if w * h < MIN_BOX_AREA or w < MIN_BOX_SIDE or h < MIN_BOX_SIDE:
                continue
            detections[track_id].append((frame_id, x, y, w, h))

    if not detections:
        print(f"  Skip {cam_id}: no valid detections in GT")
        continue

    samples = {}
    frames_needed = set()
    for track_id, dets in detections.items():
        dets.sort(key=lambda item: item[0])
        if len(dets) <= MAX_SAMPLES_PER_TRACK:
            sampled = dets
        else:
            step = len(dets) / MAX_SAMPLES_PER_TRACK
            sampled = [dets[int(index * step)] for index in range(MAX_SAMPLES_PER_TRACK)]
        samples[track_id] = sampled
        for frame_id, _, _, _, _ in sampled:
            frames_needed.add(frame_id)

    cap = cv2.VideoCapture(str(video_file))
    if not cap.isOpened():
        print(f"  Skip {cam_id}: cannot open video {video_file}")
        continue

    frame_cache = {}
    for frame_id in sorted(frames_needed):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_id - 1)
        ok, frame = cap.read()
        if ok:
            frame_cache[frame_id] = frame
    cap.release()

    camera_crop_count = 0
    for track_id, sampled_dets in samples.items():
        for frame_id, x, y, w, h in sampled_dets:
            frame = frame_cache.get(frame_id)
            if frame is None:
                continue
            frame_height, frame_width = frame.shape[:2]
            x1 = max(0, x)
            y1 = max(0, y)
            x2 = min(frame_width, x + w)
            y2 = min(frame_height, y + h)
            crop = frame[y1:y2, x1:x2]
            if crop.size == 0:
                continue
            file_name = f"{track_id:04d}_{scene}_{cam}_f{frame_id:06d}.jpg"
            crop_path = CROP_ROOT / file_name
            cv2.imwrite(str(crop_path), crop)
            all_crops.setdefault(track_id, {}).setdefault(cam_id, []).append((str(crop_path), frame_id))
            camera_crop_count += 1

    del frame_cache
    print(f"  {cam_id}: {len(detections)} vehicles, {camera_crop_count} crops extracted")

total_crops = sum(len(crops) for cameras in all_crops.values() for crops in cameras.values())
print(f"\nTotal: {len(all_crops)} vehicles, {total_crops} crops from {len(camera_dirs)} cameras")

In [ ]:
# Create train/query/gallery splits from extracted crops
import shutil

REID_ROOT = Path(CFG["dataset_root"])
if REID_ROOT.exists():
    shutil.rmtree(REID_ROOT)
for split_name in ["train", "query", "gallery"]:
    (REID_ROOT / split_name).mkdir(parents=True, exist_ok=True)

multi_cam_ids = [track_id for track_id, cams in all_crops.items() if len(cams) >= 2]
single_cam_ids = [track_id for track_id, cams in all_crops.items() if len(cams) == 1]

rng = np.random.default_rng(42)
rng.shuffle(multi_cam_ids)

n_train = int(len(multi_cam_ids) * 0.7)
train_ids = set(multi_cam_ids[:n_train])
eval_ids = set(multi_cam_ids[n_train:])

counts = {"train": 0, "query": 0, "gallery": 0}

for track_id in train_ids:
    for cam_id, items in all_crops[track_id].items():
        for crop_path, frame_id in sorted(items, key=lambda item: item[1]):
            destination = REID_ROOT / "train" / Path(crop_path).name
            shutil.copy2(crop_path, destination)
            counts["train"] += 1

for track_id in eval_ids:
    for cam_id, items in all_crops[track_id].items():
        ordered_items = sorted(items, key=lambda item: item[1])
        if not ordered_items:
            continue
        query_source, _ = ordered_items[0]
        shutil.copy2(query_source, REID_ROOT / "query" / Path(query_source).name)
        counts["query"] += 1
        for gallery_source, _ in ordered_items[1:]:
            shutil.copy2(gallery_source, REID_ROOT / "gallery" / Path(gallery_source).name)
            counts["gallery"] += 1

for track_id in single_cam_ids:
    for cam_id, items in all_crops[track_id].items():
        for crop_path, frame_id in sorted(items, key=lambda item: item[1]):
            destination = REID_ROOT / "gallery" / Path(crop_path).name
            shutil.copy2(crop_path, destination)
            counts["gallery"] += 1

print(f"Splits: train={counts['train']}, query={counts['query']}, gallery={counts['gallery']}")
print(f"Train IDs: {len(train_ids)}, Eval IDs: {len(eval_ids)}, Distractor IDs: {len(single_cam_ids)}")

In [ ]:
def parse_cityflowv2(root: str):
    train, query, gallery = [], [], []

    for split_name, split_list in [("train", train), ("query", query), ("gallery", gallery)]:
        split_dir = os.path.join(root, split_name)
        if not os.path.isdir(split_dir):
            raise FileNotFoundError(f"CityFlowV2 ReID split not found: {split_dir}")
        for fname in sorted(os.listdir(split_dir)):
            if not fname.endswith(".jpg"):
                continue
            parts = fname.split("_")
            if len(parts) < 4:
                continue
            pid = int(parts[0])
            cam_name = parts[1] + "_" + parts[2]
            split_list.append((os.path.join(split_dir, fname), pid, cam_name))

    all_cams = sorted({cam for _, _, cam in train + query + gallery})
    cam2id = {cam: idx for idx, cam in enumerate(all_cams)}
    train = [(path, pid, cam2id[cam]) for path, pid, cam in train]
    query = [(path, pid, cam2id[cam]) for path, pid, cam in query]
    gallery = [(path, pid, cam2id[cam]) for path, pid, cam in gallery]

    train_pids = sorted(set(pid for _, pid, _ in train))
    pid2label = {pid: label for label, pid in enumerate(train_pids)}
    train = [(path, pid2label[pid], cam) for path, pid, cam in train]

    return train, query, gallery, len(train_pids), len(all_cams)


train_data, query_data, gallery_data, NUM_CLASSES, NUM_CAMERAS = parse_cityflowv2(CFG["dataset_root"])
print(f"Train: {len(train_data)} images, {NUM_CLASSES} IDs, {NUM_CAMERAS} cameras")
print(f"Query: {len(query_data)}, Gallery: {len(gallery_data)}")

In [ ]:
class ReIDDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        path, pid, cam = self.data[idx]
        image = Image.open(path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, pid, cam, path


class PKSampler(Sampler):
    def __init__(self, data, p=12, k=4):
        self.p = p
        self.k = k
        self.pid_to_indices = defaultdict(list)
        for idx, (_, pid, _) in enumerate(data):
            self.pid_to_indices[pid].append(idx)
        self.pids = list(self.pid_to_indices.keys())
        self.batch_size = p * k

    def __iter__(self):
        pids = list(self.pids)
        np.random.shuffle(pids)
        batch = []
        for pid in pids:
            indices = self.pid_to_indices[pid]
            replace = len(indices) < self.k
            selected = np.random.choice(indices, size=self.k, replace=replace).tolist()
            batch.extend(selected)
            if len(batch) >= self.batch_size:
                yield from batch[: self.batch_size]
                batch = batch[self.batch_size :]
        if batch:
            yield from batch

    def __len__(self):
        return len(self.pids) * self.k


height, width = CFG["img_size"]
train_transform = T.Compose([
    T.Resize((height, width), interpolation=T.InterpolationMode.BICUBIC),
    T.RandomHorizontalFlip(p=0.5),
    T.Pad(10),
    T.RandomCrop((height, width)),
    T.ColorJitter(brightness=0.2, contrast=0.15, saturation=0.1, hue=0.05),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    T.RandomErasing(p=CFG["random_erasing_prob"], value="random"),
])
test_transform = T.Compose([
    T.Resize((height, width), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

p = CFG["batch_size"] // CFG["num_instances"]
train_loader = DataLoader(
    ReIDDataset(train_data, train_transform),
    batch_size=CFG["batch_size"],
    sampler=PKSampler(train_data, p=p, k=CFG["num_instances"]),
    num_workers=4,
    pin_memory=True,
    drop_last=True,
)
query_loader = DataLoader(
    ReIDDataset(query_data, test_transform),
    batch_size=96,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)
gallery_loader = DataLoader(
    ReIDDataset(gallery_data, test_transform),
    batch_size=96,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

print(f"Train loader: {len(train_loader)} batches, batch={CFG['batch_size']}")

In [ ]:
class IBN_a(nn.Module):
    def __init__(self, planes):
        super().__init__()
        half = planes // 2
        self.in_norm = nn.InstanceNorm2d(half, affine=True)
        self.bn_norm = nn.BatchNorm2d(planes - half)

    def forward(self, x):
        split = x.shape[1] // 2
        return torch.cat([self.in_norm(x[:, :split]), self.bn_norm(x[:, split:])], dim=1)


class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return F.adaptive_avg_pool2d(x.clamp(min=self.eps).pow(self.p), 1).pow(1.0 / self.p)


class ResNet101IBN(nn.Module):
    def __init__(self, num_classes, feat_dim=2048, last_stride=1, gem_p=3.0):
        super().__init__()
        base = tv_models.resnet101(weights=tv_models.ResNet101_Weights.DEFAULT)
        for layer in [base.layer1, base.layer2]:
            for block in layer:
                if hasattr(block, "bn1"):
                    block.bn1 = IBN_a(block.bn1.num_features)
        if last_stride == 1:
            for module in base.layer4.modules():
                if isinstance(module, nn.Conv2d) and module.stride == (2, 2):
                    module.stride = (1, 1)
        self.conv1 = base.conv1
        self.bn1 = base.bn1
        self.relu = base.relu
        self.maxpool = base.maxpool
        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        self.layer4 = base.layer4
        self.pool = GeM(p=gem_p)
        self.feat_dim = feat_dim
        self.bottleneck = nn.BatchNorm1d(feat_dim)
        self.bottleneck.bias.requires_grad_(False)
        nn.init.constant_(self.bottleneck.weight, 1)
        nn.init.constant_(self.bottleneck.bias, 0)
        self.classifier = nn.Linear(feat_dim, num_classes, bias=False)
        nn.init.normal_(self.classifier.weight, std=0.001)

    def forward(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        global_feat = self.pool(x).view(x.size(0), -1)
        bn_feat = self.bottleneck(global_feat)
        if self.training:
            return self.classifier(bn_feat), global_feat, bn_feat
        return F.normalize(bn_feat, p=2, dim=1)


model = ResNet101IBN(NUM_CLASSES, feat_dim=CFG["feat_dim"], last_stride=1, gem_p=CFG["gem_p"]).to(DEVICE)
num_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"ResNet101-IBN-a params: {num_params:.1f}M")

In [ ]:
class CrossEntropyLabelSmooth(nn.Module):
    def __init__(self, num_classes, epsilon=0.1):
        super().__init__()
        self.num_classes = num_classes
        self.epsilon = epsilon
        self.logsoftmax = nn.LogSoftmax(dim=1)

    def forward(self, inputs, targets):
        log_probs = self.logsoftmax(inputs)
        targets_oh = torch.zeros_like(log_probs).scatter_(1, targets.unsqueeze(1), 1)
        targets_smooth = (1 - self.epsilon) * targets_oh + self.epsilon / self.num_classes
        return (-targets_smooth * log_probs).mean(0).sum()


class TripletLoss(nn.Module):
    def __init__(self, margin=0.3):
        super().__init__()
        self.ranking_loss = nn.MarginRankingLoss(margin=margin)

    def forward(self, inputs, targets):
        n = inputs.size(0)
        dist = torch.cdist(inputs, inputs, p=2)
        mask = targets.unsqueeze(0).eq(targets.unsqueeze(1))
        dist_ap = torch.stack([dist[i][mask[i]].max() for i in range(n)])
        dist_an = torch.stack([dist[i][~mask[i]].min() for i in range(n)])
        return self.ranking_loss(dist_an, dist_ap, torch.ones_like(dist_an))


class CircleLoss(nn.Module):
    def __init__(self, m=0.25, gamma=80):
        super().__init__()
        self.m = m
        self.gamma = gamma
        self.soft_plus = nn.Softplus()

    def forward(self, inputs, targets):
        inputs = F.normalize(inputs, p=2, dim=1)
        n = inputs.size(0)
        sim = inputs @ inputs.t()
        mask = targets.unsqueeze(0).eq(targets.unsqueeze(1))
        op, on = 1 + self.m, -self.m
        delta_p, delta_n = 1 - self.m, self.m
        loss = 0.0
        for i in range(n):
            pos_sim = sim[i][mask[i]]
            neg_sim = sim[i][~mask[i]]
            ap = torch.clamp_min(-pos_sim.detach() + op, 0)
            an = torch.clamp_min(neg_sim.detach() + on, 0)
            logit_p = -ap * (pos_sim - delta_p) * self.gamma
            logit_n = an * (neg_sim - delta_n) * self.gamma
            loss = loss + self.soft_plus(torch.logsumexp(logit_n, 0) + torch.logsumexp(-logit_p, 0))
        return loss / n


id_loss_fn = CrossEntropyLabelSmooth(NUM_CLASSES, epsilon=CFG["label_smoothing"])
triplet_loss_fn = TripletLoss(margin=CFG["triplet_margin"])
circle_loss_fn = CircleLoss(m=CFG["circle_m"], gamma=CFG["circle_gamma"])
print("Losses configured: ID + triplet + circle")

In [ ]:
params = [
    {
        "params": list(model.conv1.parameters()) + list(model.bn1.parameters()) + list(model.layer1.parameters()) + list(model.layer2.parameters()) + list(model.layer3.parameters()) + list(model.layer4.parameters()),
        "lr": CFG["lr"] * 0.1,
    },
    {"params": model.pool.parameters(), "lr": CFG["lr"]},
    {"params": model.bottleneck.parameters(), "lr": CFG["lr"]},
    {"params": model.classifier.parameters(), "lr": CFG["lr"]},
]
optimizer = torch.optim.Adam(params, lr=CFG["lr"], weight_decay=CFG["weight_decay"])
warmup_sched = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=0.01,
    end_factor=1.0,
    total_iters=CFG["warmup_epochs"],
)
cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CFG["epochs"] - CFG["warmup_epochs"],
    eta_min=CFG["eta_min"],
)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[warmup_sched, cosine_sched],
    milestones=[CFG["warmup_epochs"]],
)
scaler = torch.amp.GradScaler("cuda") if CFG["fp16"] and torch.cuda.is_available() else None
print(f"Optimizer ready. LR={CFG['lr']}, scheduler=cosine, warmup={CFG['warmup_epochs']}")

In [ ]:
@torch.no_grad()
def extract_features(model, loader, device):
    model.eval()
    feats, pids, cams = [], [], []
    for imgs, pid, cam, _ in loader:
        imgs = imgs.to(device)
        feats_main = model(imgs)
        feats_flip = model(torch.flip(imgs, dims=[3]))
        feats_avg = F.normalize((feats_main + feats_flip) / 2, p=2, dim=1)
        feats.append(feats_avg.cpu().numpy())
        pids.append(pid.numpy())
        cams.append(cam.numpy())
    return np.concatenate(feats), np.concatenate(pids), np.concatenate(cams)


def eval_reid(qf, qp, qc, gf, gp, gc, max_rank=50):
    dist = 1 - qf @ gf.T
    all_ap = []
    all_cmc = []
    for i in range(dist.shape[0]):
        order = np.argsort(dist[i])
        valid = ~((gp[order] == qp[i]) & (gc[order] == qc[i]))
        matches = (gp[order][valid] == qp[i]).astype(np.int32)
        if matches.sum() == 0:
            continue
        cmc = matches.cumsum()
        cmc = (cmc > 0).astype(np.float32)
        cmc_vec = np.zeros(max_rank, dtype=np.float32)
        upto = min(max_rank, len(cmc))
        cmc_vec[:upto] = cmc[:upto]
        all_cmc.append(cmc_vec)
        cum_tp = matches.cumsum()
        precision = cum_tp / (np.arange(len(matches)) + 1)
        all_ap.append((precision * matches).sum() / matches.sum())
    mAP = float(np.mean(all_ap)) if all_ap else 0.0
    cmc = np.mean(all_cmc, axis=0) if all_cmc else np.zeros(max_rank, dtype=np.float32)
    return mAP, cmc

In [ ]:
best_mAP = 0.0
history = []

for epoch in range(CFG["epochs"]):
    model.train()
    running = {"loss": 0.0, "id": 0.0, "tri": 0.0, "cir": 0.0, "n": 0}
    start_time = time.time()

    for imgs, pids, _, _ in train_loader:
        imgs = imgs.to(DEVICE)
        pids = pids.to(DEVICE).long()
        optimizer.zero_grad()

        with torch.amp.autocast("cuda", enabled=scaler is not None):
            cls_score, global_feat, bn_feat = model(imgs)
            loss_id = id_loss_fn(cls_score, pids) * CFG["id_weight"]
            loss_tri = triplet_loss_fn(global_feat, pids) * CFG["triplet_weight"]
            loss_cir = circle_loss_fn(global_feat, pids) * CFG["circle_weight"]
            loss = loss_id + loss_tri + loss_cir

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        running["loss"] += loss.item()
        running["id"] += loss_id.item()
        running["tri"] += loss_tri.item()
        running["cir"] += loss_cir.item()
        running["n"] += 1

    scheduler.step()
    n_batches = max(running["n"], 1)
    elapsed = time.time() - start_time
    lr = optimizer.param_groups[0]["lr"]
    print(
        f"E{epoch:02d} | {elapsed:.0f}s | LR={lr:.6f} | "
        f"L={running['loss']/n_batches:.4f} ID={running['id']/n_batches:.4f} "
        f"Tri={running['tri']/n_batches:.4f} Cir={running['cir']/n_batches:.4f}"
    )

    if (epoch + 1) % CFG["eval_every"] == 0 or epoch == CFG["epochs"] - 1:
        qf, qp, qc = extract_features(model, query_loader, DEVICE)
        gf, gp, gc = extract_features(model, gallery_loader, DEVICE)
        mAP, cmc = eval_reid(qf, qp, qc, gf, gp, gc)
        print(f"  >> mAP={mAP:.4f}, R1={cmc[0]:.4f}, R5={cmc[4]:.4f}")
        history.append({"epoch": epoch, "mAP": float(mAP), "R1": float(cmc[0])})

        torch.save(
            {
                "state_dict": model.state_dict(),
                "epoch": epoch,
                "optimizer": optimizer.state_dict(),
                "mAP": float(mAP),
            },
            f"{CFG['checkpoint_dir']}/ckpt_e{epoch:02d}.pth",
        )

        if mAP > best_mAP:
            best_mAP = mAP
            torch.save({"state_dict": model.state_dict(), "epoch": epoch, "mAP": float(mAP)}, CFG["weights_output"])
            print(f"  * New best mAP: {best_mAP:.4f}")

print(f"Training complete. Best mAP: {best_mAP:.4f}")

In [ ]:
ckpt = torch.load(CFG["weights_output"], map_location="cpu", weights_only=False)
model.load_state_dict(ckpt["state_dict"])
model.to(DEVICE)

qf, qp, qc = extract_features(model, query_loader, DEVICE)
gf, gp, gc = extract_features(model, gallery_loader, DEVICE)
mAP, cmc = eval_reid(qf, qp, qc, gf, gp, gc)
print(f"Final: mAP={mAP:.4f}, R1={cmc[0]:.4f}, R5={cmc[4]:.4f}, R10={cmc[9]:.4f}")
print(json.dumps(history, indent=2))

In [ ]:
import shutil

output_name = "resnet101ibn_cityflowv2_384px_best.pth"
shutil.copy2(CFG["weights_output"], f"/kaggle/working/{output_name}")
print(f"Weights saved: /kaggle/working/{output_name}")
print(f"Size: {os.path.getsize(f'/kaggle/working/{output_name}') / 1e6:.1f} MB")

with open("/kaggle/working/training_history_resnet101ibn.json", "w", encoding="utf-8") as handle:
    json.dump({"config": CFG, "history": history, "best_mAP": float(best_mAP)}, handle, indent=2, default=str)

In [ ]:
ckpt = torch.load("/kaggle/working/resnet101ibn_cityflowv2_384px_best.pth", map_location="cpu", weights_only=False)
keys = list(ckpt["state_dict"].keys())
print(f"State dict keys: {len(keys)}")
print(f"Key samples: {keys[:5]} ... {keys[-5:]}")
assert any(key.startswith("conv1.") for key in keys), "Missing conv1 weights"
assert any(key.startswith("layer1.") for key in keys), "Missing layer1 weights"
assert any(key.startswith("pool.") for key in keys), "Missing GeM weights"
assert any(key.startswith("bottleneck.") for key in keys), "Missing BNNeck weights"
print("Checkpoint structure validated")